# PepFoundry

This example shows how to initialize the **PepFoundry** interface, which can be used to design and analyze peptides.


In [2]:
from pepfoundry.interface import PepFoundry

pepfoundry = PepFoundry() 

# Save Peptide Molecules Example

This script reads peptide sequences from a CSV file, generates RDKit molecule objects using **PepFoundry**, and caches them as pickle files to avoid repeated computation.


In [3]:
import os
import pickle
import pandas as pd

# ======================
# Create a cache directory if it doesn't exist
# ======================
cache_dir = "cache_molecule_files"
os.makedirs(cache_dir, exist_ok=True)

# ======================
# Save molecules (with automatic generation using pepfoundry)
# ======================

def save_molecules(csv_path, cache_dir="cache_molecule_files"):
    """
    Reads peptide sequences from a CSV file, generates RDKit molecules using pepfoundry,
    and saves them as a pickle file inside the cache directory.
    Skips generation if the pickle already exists.
    
    Parameters:
        csv_path : str
            Path to the CSV file containing peptide sequences (first column).
        cache_dir : str
            Directory to store the pickle file.
    
    Returns:
        molecules : list
            List of RDKit molecule objects.
    """
    os.makedirs(cache_dir, exist_ok=True)
    
    # Create filename based on CSV name
    base_name = os.path.basename(csv_path).replace(".csv", "_molecules.pkl")
    file_path = os.path.join(cache_dir, base_name)
    
    # Check if file already exists
    if os.path.exists(file_path):
        print(f"Molecule pickle already exists at {file_path}. Loading...")
        with open(file_path, "rb") as f:
            molecules = pickle.load(f)
        return molecules
    
    # Read sequences from CSV
    df = pd.read_csv(csv_path)
    sequences = df.iloc[:, 0].astype(str)
    
    print(f"Generating molecules for {csv_path} using PepFoundry...")
    molecules = [pepfoundry.get_peptide(seq, plot_peptide=False) for seq in sequences]
    
    # Save molecules to pickle
    with open(file_path, "wb") as f:
        pickle.dump(molecules, f)
    print(f"Molecules saved to {file_path}")
    
    


In [5]:
dataset_training_molecules = save_molecules("datasets/testing_AmpHGT.csv", cache_dir)
dataset_testing_molecules = save_molecules("datasets/training_p_21865_n_39854_dataset_AmpHGT.csv", cache_dir)


Molecule pickle already exists at cache_molecule_files/testing_AmpHGT_molecules.pkl. Loading...
Molecule pickle already exists at cache_molecule_files/training_p_21865_n_39854_dataset_AmpHGT_molecules.pkl. Loading...


# PeptideDataset Class

The `PeptideDataset` class provides tools to load peptide sequences from a CSV, manage molecules, compute molecular fingerprints, and generate one-hot encodings.  
It supports caching to avoid recomputation of molecules, fingerprints, and encodings.

## Features

1. **Load CSV and molecules**
   - Reads peptide sequences (first column) and targets (second column) from a CSV file.
   - Loads pre-generated molecules from a cache directory (`molecule_dir`).

2. **Caching**
   - Stores targets, fingerprints, and one-hot encodings in `cache_dir` to speed up future runs.

3. **Molecular fingerprints**
   - **Morgan fingerprints:** configurable radius and bit size, with caching.
   - **MACCS keys:** generates 167-bit MACCS fingerprints, with caching.

4. **Sequence encoding**
   - Tokenizes peptide sequences (supports non-natural amino acids with `{}` notation).
   - Computes one-hot encodings using `MultiLabelBinarizer`.
   - Supports reusing classes for test datasets.

In [9]:
import os
import pickle
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem, MACCSkeys, DataStructs
import re
from sklearn.preprocessing import MultiLabelBinarizer

class PeptideDataset:
    def __init__(self, csv_path, cache_dir="cache_fingerprints", molecule_dir="cache_molecule_files"):
        """
        Load CSV and prepare peptide molecules.
        Molecules are loaded from 'molecule_dir' instead of generating.
        Supports caching of fingerprints, one-hot encodings, and targets.
        """
        self.csv_path = csv_path
        self.cache_dir = cache_dir
        self.molecule_dir = molecule_dir
        os.makedirs(self.cache_dir, exist_ok=True)
        os.makedirs(self.molecule_dir, exist_ok=True)
        
        # Load CSV
        self.df = pd.read_csv(csv_path)
        self.sequences = self.df.iloc[:, 0].astype(str)
        
        # Load targets (second column)
        self.targets = self.df.iloc[:, 1].values
        # Save cache for targets
        target_cache_file = os.path.join(self.cache_dir, os.path.basename(csv_path).replace(".csv", "_targets.npy"))
        if not os.path.exists(target_cache_file):
            np.save(target_cache_file, self.targets)
            print(f"Targets saved to {target_cache_file}")
        else:
            self.targets = np.load(target_cache_file)
            print(f"Targets loaded from {target_cache_file}")
        
        # Load molecules from molecule_dir
        mol_cache_file = os.path.join(self.molecule_dir, os.path.basename(csv_path).replace(".csv", "_molecules.pkl"))
        if os.path.exists(mol_cache_file):
            with open(mol_cache_file, "rb") as f:
                self.molecules = pickle.load(f)
            print(f"Molecules loaded from {mol_cache_file}")
        else:
            raise FileNotFoundError(f"Molecule file not found in {mol_cache_file}. Please generate and save molecules first.")
        
        # Internal caches
        self._morgan_fp = {}       # <--- cache independiente para cada config
        self._maccs_fp = None
        self._onehot = None
        self._onehot_classes = None
        
    def _get_cache_path(self, name):
        base = os.path.basename(self.csv_path).replace(".csv", "")
        return os.path.join(self.cache_dir, f"{base}_{name}.npy")
    
    # -------------------
    # Morgan fingerprints genérico
    # -------------------
    def get_morgan_fingerprints(self, radius=2, nBits=1024):
        """
        Genera o carga Morgan fingerprints con un radius y nBits dados.
        Devuelve un array (n_mols, nBits).
        """
        key = f"r{radius}_nBits{nBits}"
        cache_file = self._get_cache_path(f"morgan_{key}")
        
        # Si ya está en memoria
        if key in self._morgan_fp:
            return self._morgan_fp[key]
        
        # Si está en disco
        if os.path.exists(cache_file):
            arr = np.load(cache_file)
            self._morgan_fp[key] = arr
            print(f"Morgan fingerprints loaded from {cache_file}")
            return arr
        
        # Generar
        print(f"Generating Morgan fingerprints (radius={radius}, nBits={nBits})...")
        fps = [AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nBits) for mol in self.molecules]
        arrs = np.zeros((len(fps), nBits), dtype=np.uint8)
        for i, fp in enumerate(fps):
            DataStructs.ConvertToNumpyArray(fp, arrs[i])
        np.save(cache_file, arrs)
        self._morgan_fp[key] = arrs
        print(f"Morgan fingerprints saved to {cache_file}")
        return arrs
    
    # -------------------
    # MACCS keys
    # -------------------
    def get_maccs_keys(self):
        cache_file = self._get_cache_path("maccs")
        if os.path.exists(cache_file):
            self._maccs_fp = np.load(cache_file)
            print(f"MACCS keys loaded from {cache_file}")
        elif self._maccs_fp is None:
            print("Generating MACCS keys...")
            fps = [MACCSkeys.GenMACCSKeys(mol) for mol in self.molecules]
            arrs = np.zeros((len(fps), 167), dtype=np.uint8)
            for i, fp in enumerate(fps):
                DataStructs.ConvertToNumpyArray(fp, arrs[i])
            self._maccs_fp = arrs
            np.save(cache_file, self._maccs_fp)
            print(f"MACCS keys saved to {cache_file}")
        return self._maccs_fp
    
    # -------------------
    # Tokenize sequences
    # -------------------
    def _tokenize_sequence(self, seq):
        return re.findall(r'\{.*?\}|.', seq)
    
    def get_one_hot_encoding(self, classes=None):
        """
        Compute one-hot encoding.
        
        Parameters:
            classes : array-like or None
                If provided, uses these classes to transform sequences (useful for testing set).
                If None, fits MultiLabelBinarizer on current sequences (training set).
        
        Returns:
            onehot : list of arrays
            classes_ : list of tokens/classes
        """
        cache_file = self._get_cache_path("onehot.npy")
        classes_file = self._get_cache_path("onehot_classes.npy")
        
        if classes is None and os.path.exists(cache_file) and os.path.exists(classes_file):
            self._onehot = np.load(cache_file, allow_pickle=True)
            self._onehot_classes = np.load(classes_file, allow_pickle=True)
            print(f"One-hot encoding loaded from {cache_file}")
        else:
            tokenized_seqs = [self._tokenize_sequence(seq) for seq in self.sequences]
            if classes is not None:
                mlb = MultiLabelBinarizer(classes=classes)
                mlb.fit([classes])
            else:
                mlb = MultiLabelBinarizer()
                mlb.fit(tokenized_seqs)
            onehots = [mlb.transform([seq_tokens]) for seq_tokens in tokenized_seqs]
            self._onehot = onehots
            self._onehot_classes = mlb.classes_
            np.save(cache_file, self._onehot)
            np.save(classes_file, self._onehot_classes)
            print(f"One-hot encoding saved to {cache_file}")
        
        return self._onehot, self._onehot_classes


# Compute One-Hot Classes

The `compute_onehot_classes` function reads peptide sequences from training and testing CSV files, tokenizes them, and computes the unique tokens for one-hot encoding.  
This ensures that both datasets share the same encoding scheme.

## Functionality
1. **Tokenization**
   - Splits sequences into individual characters.
   - Keeps `{…}` blocks (for non-natural amino acids) as single tokens.

2. **CSV Input**
   - Reads sequences from the first column of training and testing CSVs.

3. **Compute Classes**
   - Uses `MultiLabelBinarizer` to determine all unique tokens across both datasets.
   - Returns a NumPy array of token classes suitable for one-hot encoding.


In [6]:
import pandas as pd
import re
from sklearn.preprocessing import MultiLabelBinarizer

def compute_onehot_classes(training_csv, testing_csv):
    """
    Reads training and testing CSVs, tokenizes all sequences, 
    and computes the unique token classes to be used for one-hot encoding.
    
    Parameters:
        training_csv : str
            Path to the training CSV file.
        testing_csv : str
            Path to the testing CSV file.
    
    Returns:
        classes : np.ndarray
            Array of unique tokens from both datasets.
    """
    def tokenize_sequence(seq):
        # Keep {…} as a single token
        return re.findall(r'\{.*?\}|.', seq)
    
    # Read CSVs
    df_train = pd.read_csv(training_csv)
    df_test = pd.read_csv(testing_csv)
    
    sequences_train = df_train.iloc[:, 0].astype(str)
    sequences_test = df_test.iloc[:, 0].astype(str)
    
    # Tokenize all sequences
    tokenized_train = [tokenize_sequence(seq) for seq in sequences_train]
    tokenized_test = [tokenize_sequence(seq) for seq in sequences_test]
    
    # Fit MultiLabelBinarizer on both sets to get all unique tokens
    mlb = MultiLabelBinarizer()
    mlb.fit(tokenized_train + tokenized_test)
    
    return mlb.classes_


In [7]:
classes = compute_onehot_classes(
    "datasets/training_p_21865_n_39854_dataset_AmpHGT.csv",
    "datasets/testing_AmpHGT.csv"
)

print("Number of unique tokens:", len(classes))
print("Tokens:", classes)

Number of unique tokens: 57
Tokens: ['A' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'K' 'L' 'M' 'N' 'P' 'Q' 'R' 'S' 'T' 'V'
 'W' 'Y' 'a' 'f' 'h' 'i' 'k' 'l' 'm' 'p' 'r' 's' 'v' 'w' 'y'
 '{3&4-dihydrox-F}' '{4&5-hydrox-L}' '{4-hydrox-P}' '{4-hydrox-p}'
 '{6-Br-W}' '{GlcNAc-S}' '{GlcNAc-T}' '{GlcNAc-asn}' '{M-sulfo}'
 '{S-Glc-C}' '{allo-i}' '{asy-dime-R}' '{dime-A}' '{formyl-M}' '{iso-D}'
 '{iso-Q}' '{k-ac}' '{phospho-S}' '{phospho-Y}' '{pyro-E}' '{seleno-C}'
 '{succinyl-K}' '{sulfo-Y}' '{trime-L}']


In [10]:
dataset_training = PeptideDataset(csv_path="datasets/training_p_21865_n_39854_dataset_AmpHGT.csv")
y_train = dataset_training.targets
dataset_training.get_morgan_fingerprints(radius=2, nBits=2048)
dataset_training.get_morgan_fingerprints(radius=2, nBits=1024)
dataset_training.get_maccs_keys()
dataset_training.get_one_hot_encoding(classes=classes)


dataset_testing = PeptideDataset(csv_path="datasets/testing_AmpHGT.csv")
y_test = dataset_testing.targets
dataset_testing.get_morgan_fingerprints(radius=2, nBits=2048)
dataset_testing.get_morgan_fingerprints(radius=2, nBits=1024)
dataset_testing.get_maccs_keys()
dataset_testing.get_one_hot_encoding(classes=classes)

Targets loaded from cache_fingerprints/training_p_21865_n_39854_dataset_AmpHGT_targets.npy
Molecules loaded from cache_molecule_files/training_p_21865_n_39854_dataset_AmpHGT_molecules.pkl
Morgan fingerprints loaded from cache_fingerprints/training_p_21865_n_39854_dataset_AmpHGT_morgan_r2_nBits2048.npy
Morgan fingerprints loaded from cache_fingerprints/training_p_21865_n_39854_dataset_AmpHGT_morgan_r2_nBits1024.npy
MACCS keys loaded from cache_fingerprints/training_p_21865_n_39854_dataset_AmpHGT_maccs.npy
One-hot encoding saved to cache_fingerprints/training_p_21865_n_39854_dataset_AmpHGT_onehot.npy.npy
Targets loaded from cache_fingerprints/testing_AmpHGT_targets.npy
Molecules loaded from cache_molecule_files/testing_AmpHGT_molecules.pkl
Morgan fingerprints loaded from cache_fingerprints/testing_AmpHGT_morgan_r2_nBits2048.npy
Morgan fingerprints loaded from cache_fingerprints/testing_AmpHGT_morgan_r2_nBits1024.npy
MACCS keys loaded from cache_fingerprints/testing_AmpHGT_maccs.npy
One-h

([array([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0,
          0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
          0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]),
  array([[1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0,
          0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
          0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]),
  array([[1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0,
          0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
          0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]),
  array([[1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0,
          0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
          0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]),
  array([[1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0,
          0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
          0, 0, 0, 0, 0, 0, 0, 0

# Peptide Dataset Feature Extraction Example

This script demonstrates how to extract features and inspect the dataset sizes for training and testing peptide datasets using the `PeptideDataset` class.

## Steps

1. **Target Sizes**
   - Prints the number of samples (targets) in the training and testing datasets.

2. **Morgan Fingerprints**
   - Computes Morgan fingerprints with 1024 and 2048 bits for both datasets.
   - Prints the shapes of the resulting fingerprint arrays.

3. **MACCS Keys**
   - Generates MACCS keys (167-bit fingerprints) for training and testing sets.
   - Prints the shapes of the MACCS arrays.

4. **One-Hot Encoding**
   - Computes one-hot encodings for peptide sequences.
   - Combines individual sequence encodings into full arrays.
   - Prints the shapes of the one-hot arrays and the number of unique tokens/classes.


In [ ]:
# --------------------------
# Target sizes
# --------------------------
print("Number of samples (targets):")
print("Training:", len(dataset_training.targets))
print("Testing:", len(dataset_testing.targets))

# --------------------------
# Morgan fingerprints
# --------------------------
morgan_fp1024_training = dataset_training.get_morgan_fingerprints(radius=2, nBits=1024)
morgan_fp1024_testing = dataset_testing.get_morgan_fingerprints(radius=2, nBits=1024)

print("\nMorgan fingerprints 1024 shapes:")
print("Training:", morgan_fp1024_training.shape)
print("Testing:", morgan_fp1024_testing.shape)

morgan_fp2048_training = dataset_training.get_morgan_fingerprints(radius=2, nBits=2048)
morgan_fp2048_testing = dataset_testing.get_morgan_fingerprints(radius=2, nBits=2048)

print("\nMorgan fingerprints 2048 shapes:")
print("Training:", morgan_fp2048_training.shape)
print("Testing:", morgan_fp2048_testing.shape)

# --------------------------
# MACCS keys
# --------------------------
maccs_fp_training = dataset_training.get_maccs_keys()
maccs_fp_testing = dataset_testing.get_maccs_keys()

print("\nMACCS keys shapes:")
print("Training:", maccs_fp_training.shape)
print("Testing:", maccs_fp_testing.shape)

# --------------------------
# One-hot encoding
# --------------------------
onehot_training, classes_training = dataset_training.get_one_hot_encoding()
onehot_testing, classes_testing = dataset_testing.get_one_hot_encoding()
onehot_training_array = np.vstack([x for x in onehot_training])
onehot_testing_array  = np.vstack([x for x in onehot_testing])


print("\nOne-hot encoding shapes:")
print("Training:", onehot_training_array.shape)
print("Testing:", onehot_testing_array.shape)
print("Number of tokens/classes:", len(classes_training))


# Machine Learning Model Training and Hyperparameter Tuning

This script performs hyperparameter tuning, training, and evaluation of multiple machine learning models on peptide datasets using different feature sets.

## Steps

1. **Hyperparameter Grids**
   - Defines parameter grids for SVM, RandomForest, DecisionTree, ExtraTrees, GradientBoosting, kNN, and MLP.

2. **Model Initialization**
   - Creates instances of each model, setting random seeds where applicable for reproducibility.

3. **Feature Sets**
   - Uses three types of features: 
     - Morgan fingerprints
     - MACCS keys
     - One-hot encodings

4. **Results Storage**
   - Prepares a `DataFrame` to store Accuracy, F1, Avg. Precision, and ROC-AUC scores for each model-feature combination.
   - Stores best hyperparameters in a dictionary.

5. **RandomizedSearchCV Loop**
   - Splits training data into train/validation sets.
   - Performs randomized hyperparameter search for models with defined grids.
   - Trains models and evaluates them on the test set.
   - Records metrics in the results DataFrame.

6. **Evaluation Metrics**
   - Accuracy, F1-score, Avg. Precision, ROC-AUC.
   - Handles models that do not provide probability predictions gracefully.

7. **Save Results**
   - Saves evaluation metrics to `model_evaluation_results_finetuned.xlsx`.
   - Saves best hyperparameters to `best_params.xlsx`.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import accuracy_score, f1_score, average_precision_score, roc_auc_score
import numpy as np

# ==========================
# Hyperparameter grids
# ==========================
param_grids = {
    "SVM": {
        'C': [0.1, 1, 10],
        'gamma': ['scale', 0.1, 0.01],
        'kernel': ['rbf', 'poly', 'sigmoid']
    },
    "RandomForest": {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5, 10]
    },
    "DecisionTree": {
        'max_depth': [None, 5, 10, 20],
        'min_samples_split': [2, 5, 10],
        'criterion': ['gini', 'entropy']
    },
    "ExtraTrees": {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5, 10]
    },
    "GradientBoosting": {
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.1, 0.05, 0.01],
        'max_depth': [3, 5, 7]
    },
    "kNN": {
        'n_neighbors': [3, 5, 10],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan', 'minkowski']
    },
    "MLP": {
        'hidden_layer_sizes': [(128,), (128, 64), (256, 128, 64)],
        'activation': ['relu', 'tanh', 'logistic'],
        'alpha': [0.0001, 0.001, 0.01],
        'learning_rate_init': [0.001, 0.01, 0.1]
    }
}

# --------------------------
# Models
# --------------------------
seed = 42
models = {
    "SVM": SVC(probability=True, random_state=seed),
    "RandomForest": RandomForestClassifier(random_state=seed),
    "DecisionTree": DecisionTreeClassifier(random_state=seed),
    "ExtraTrees": ExtraTreesClassifier(random_state=seed),
    "GradientBoosting": GradientBoostingClassifier(random_state=seed),
    "kNN": KNeighborsClassifier(),  # deterministic, no seed needed
    "MLP": MLPClassifier(random_state=seed)
}

# --------------------------
# Feature sets
# --------------------------
feature_sets = {
    "Morgan fingerprints": (morgan_fp_training, morgan_fp_testing),
    "MACCS keys":          (maccs_fp_training, maccs_fp_testing),
    "One-hot encoding":    (onehot_training_array, onehot_testing_array)
}

# ==========================
# DataFrame to store results
# ==========================
metrics = ["Accuracy", "F1", "Precision", "ROC-AUC"]
results = pd.DataFrame(index=feature_sets.keys(),
                       columns=pd.MultiIndex.from_product([models.keys(), metrics]))
best_params = {}

# ==========================
# RandomizedSearch loop
# ==========================
for feat_name, (X_train_full, X_test) in feature_sets.items():
    print(f"\n=== Feature set: {feat_name} ===")
    
    # Split train en 80% train / 20% validación
    X_train, X_val, y_train_split, y_val_split = train_test_split(
        X_train_full, y_train, test_size=0.2, random_state=42, stratify=y_train
    )

    for model_name, model in models.items():
        print(f"-> Tuning {model_name} ...")
        if model_name in param_grids:
            # Para MLP activamos early_stopping + validation_fraction=0.2
            if model_name == "MLP":
                model.set_params(early_stopping=True, validation_fraction=0.2)
                X_train = X_train_full
                y_train_split = y_train
            
            random_search = RandomizedSearchCV(
                                                estimator=model,
                                                param_distributions=param_grids[model_name],
                                                n_iter=10,        # nº de combinaciones al azar
                                                cv=3,
                                                scoring='f1',
                                                n_jobs=1,
                                                random_state=42
                                            )
            random_search.fit(X_train, y_train_split)
            best_model = random_search.best_estimator_
            best_params[(feat_name, model_name)] = random_search.best_params_
        else:
            best_model = model.fit(X_train, y_train_split)
            best_params[(feat_name, model_name)] = "default"
        
        # Evaluate on test set
        y_pred = best_model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        f1  = f1_score(y_test, y_pred)
        avg_prec = average_precision_score(y_test, y_pred) 
        try:
            y_prob = best_model.predict_proba(X_test)[:, 1]
            auc = roc_auc_score(y_test, y_prob)
        except:
            auc = None
        
        results.loc[feat_name, (model_name, "Accuracy")]  = round(acc, 3)
        results.loc[feat_name, (model_name, "F1")]        = round(f1, 3)
        results.loc[feat_name, (model_name, "Avg. Precision")] = round(avg_prec, 3)
        results.loc[feat_name, (model_name, "ROC-AUC")]   = round(auc, 3) if auc is not None else None

# ==========================
# Save results
# ==========================
results.to_excel("model_evaluation_results_finetuned.xlsx")
pd.DataFrame.from_dict(best_params, orient='index').to_excel("best_params.xlsx")

print("\nResults saved to 'model_evaluation_results_finetuned.xlsx'")
print("Best parameters saved to 'best_params.xlsx'")


# Machine Learning Models with Tuned Hyperparameters

This script defines multiple machine learning models with the best hyperparameters obtained from prior tuning.  
It also sets up feature sets for evaluation.

## Feature Sets
- **Morgan fingerprints 1024 bits**
- **Morgan fingerprints 2048 bits**
- **MACCS keys** (167-bit fingerprints)
- **One-hot encoding** of peptide sequences

## Models with Best Parameters
- **RandomForestClassifier:** n_estimators=100, min_samples_split=2  
- **DecisionTreeClassifier:** max_depth=20, min_samples_split=2, criterion='entropy'  
- **ExtraTreesClassifier:** n_estimators=100, min_samples_split=2  
- **GradientBoostingClassifier:** n_estimators=300, learning_rate=0.1, max_depth=7  
- **kNN:** n_neighbors=10, weights='distance', metric='manhattan'  
- **MLPClassifier:** hidden layers (128,64), activation='relu', alpha=0.001, learning_rate_init=0.001, max_iter=500  
- **SVM:** kernel='rbf', C=10, gamma=0.1, probability=True  

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

# --------------------------
# Feature sets
# --------------------------
feature_sets = {
    "Morgan fingerprints 1024": (morgan_fp1024_training, morgan_fp1024_testing),
    "Morgan fingerprints 2048": (morgan_fp2048_training, morgan_fp2048_testing),
    "MACCS keys":          (maccs_fp_training, maccs_fp_testing),
    "One-hot encoding":    (onehot_training_array, onehot_testing_array)
}

# --------------------------
# Models ; Best parameters from hyperparameter tuning
# --------------------------
seed = 42
models = {
    "RandomForest": RandomForestClassifier(n_estimators=100, min_samples_split=2,random_state=seed),
    "DecisionTree": DecisionTreeClassifier(max_depth=20,min_samples_split=2,criterion='entropy', random_state=seed),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=100, min_samples_split=2, random_state=seed),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=300, learning_rate=0.1,max_depth=7, random_state=seed),
    "kNN": KNeighborsClassifier(weights='distance', metric = 'manhattan', n_neighbors=10),  # deterministic, no seed needed
    "MLP": MLPClassifier(
                        hidden_layer_sizes=(128, 64),
                        activation='relu',
                        solver='adam',           
                        learning_rate_init=1e-3,
                        alpha=0.001,             
                        max_iter=500,
                        random_state=seed
                    ),
    "SVM": SVC(kernel='rbf', C=10, gamma=0.1, probability=True, random_state=seed),
}



# Model Training and Evaluation

## Overview
This script trains multiple machine learning models on different peptide feature sets and evaluates their performance using key metrics.

## Steps

1. **Initialize Results DataFrame**
   - Stores metrics (Accuracy, F1-score, Average Precision, ROC-AUC) for each model-feature combination.

2. **Training & Evaluation Loop**
   - Iterates over each feature set and each model.
   - Trains the model on the training data (`X_train`, `y_train`).
   - Predicts on the test data (`X_test`).

3. **Compute Metrics**
   - **Accuracy**: proportion of correct predictions.
   - **F1-score**: harmonic mean of precision and recall.
   - **Average Precision**: precision-recall metric for binary classification.
   - **ROC-AUC**: area under the ROC curve, if probability predictions are available.

4. **Store and Print Results**
   - Rounds metric values to 3 decimals and prints progress.
   - Stores results in a hierarchical DataFrame.

5. **Save Results**
   - Exports the results to `model_evaluation_results.xlsx` for further analysis.


In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, average_precision_score, roc_auc_score

# Initialize a DataFrame to store results
metrics = ["Accuracy", "F1", "AvgPrecision", "ROC-AUC"]
results = pd.DataFrame( index=feature_sets.keys(),
                        columns=pd.MultiIndex.from_product([models.keys(), metrics]))

# Count total iterations for progress tracking
total_iterations = len(feature_sets) * len(models)
iteration = 0

# Training & evaluation loop
for feat_name, (X_train, X_test) in feature_sets.items():
    print(f"\n=== Feature set: {feat_name} ===")
    
    for model_name, model in models.items():
        iteration += 1
        print(f"[{iteration}/{total_iterations}] Training {model_name}...")
        
        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Compute metrics
        acc  = accuracy_score(y_test, y_pred)
        f1   = f1_score(y_test, y_pred)
        
        # Average Precision (binary classification)
        try:
            y_prob = model.predict_proba(X_test)[:, 1]
            avg_prec = average_precision_score(y_test, y_prob)
            auc = roc_auc_score(y_test, y_prob)
        except:
            avg_prec = None
            auc = None
        
        # Store results, rounding to 3 decimals
        results.loc[feat_name, (model_name, "Accuracy")]     = round(acc, 3)
        results.loc[feat_name, (model_name, "F1")]           = round(f1, 3)
        results.loc[feat_name, (model_name, "AvgPrecision")] = round(avg_prec, 3) if avg_prec is not None else None
        results.loc[feat_name, (model_name, "ROC-AUC")]      = round(auc, 3) if auc is not None else None
        
        print(f"Done: Accuracy={acc:.3f}, F1={f1:.3f}, AvgPrecision={avg_prec if avg_prec is None else round(avg_prec,3)}, ROC-AUC={auc if auc is None else round(auc,3)}")

# Save to Excel
results.to_excel("model_evaluation_results.xlsx")
print("\nAll results saved to 'model_evaluation_results.xlsx'")
